In [3]:
# 프롬프트 1로 Codex(GPT-6.1 Sol High)가 생성한 코드
import re


def read_bnc_tagged(filename="/content/bnc_tagged.txt"):
    results = []
    tokens = []
    last_id = None
    current_id = None
    sequence = 1
    missing_start_count = 0

    start_pattern = re.compile(r"""<s\s+id\s*=\s*["']?(\d+)["']?\s*>""")

    with open(filename, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.rstrip("\r\n")
            tag = line.strip()

            # 시작 태그에서 숫자 ID 추출
            match = start_pattern.fullmatch(tag)
            if match:
                last_id = match.group(1)
                current_id = last_id
                sequence = 1
                continue

            # 종료 태그를 만나면 문장 저장
            if tag == "</s>":
                if current_id is None:
                    if last_id is None:
                        raise ValueError(
                            "첫 문장에 시작 태그가 없어 기준 ID를 정할 수 없습니다."
                        )

                    sequence += 1
                    current_id = f"{last_id}-{sequence}"
                    missing_start_count += 1

                results.append((current_id, tokens))
                tokens = []
                current_id = None
                continue

            # 정확히 7개 필드인 토큰만 저장
            fields = line.split("\t")
            if len(fields) == 7:
                tokens.append(fields)

    print(f"전체 문장 수: {len(results)}")
    print(f"시작 태그가 없던 문장 수: {missing_start_count}")

    return results


# 실행: 반환된 결과 리스트를 sentences에 저장
sentences = read_bnc_tagged()

전체 문장 수: 94171
시작 태그가 없던 문장 수: 2598


In [6]:
# 프롬프트 2로 Codex(GPT-6.1 Sol High)가 생성한 코드
def read_noun_list(filename="/content/noun_list.txt"):
    noun_list = {}

    with open(filename, "r", encoding="utf-8-sig") as f:
        for line in f:
            if not line.strip():
                continue

            lemma = line.split("\t")[0].strip().lower()
            noun_list[lemma] = []

    return noun_list


# 실행: 반환된 딕셔너리를 noun_list에 저장
noun_list = read_noun_list()

In [7]:
# 프롬프트 3으로 Codex(GPT-6.1 Sol High)가 생성한 코드
def extract_noun_examples(corpus_data, noun_list):
    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        for token in tokens:
            lemma = token[2].lower()
            upos = token[1]

            if upos == "NOUN" and lemma in noun_list:
                noun_list[lemma].append(
                    (sentence_id, "-", sentence_text)
                )

    return noun_list


# 실행: 명사별 용례를 noun_list에 추가
noun_list = extract_noun_examples(sentences, noun_list)

In [9]:
# 프롬프트 4로 Codex(GPT-6.1 Sol High)가 생성한 코드
def extract_possessive_examples(corpus_data, noun_list):
    possessives = {"my", "your", "his", "her", "its", "our", "their"}

    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사를 핵어로 갖는 의존소 확인
            for dependent in dependents.get(token[3], []):
                if (
                    dependent[6] == "nmod:poss(소유 수식어)"
                    and dependent[0].lower() in possessives
                ):
                    noun_list[lemma].append(
                        (sentence_id, dependent[0], sentence_text)
                    )

    return noun_list


# 빈 용례 목록으로 시작하여 이전 추출 결과와 섞이지 않도록 함
possessive_noun_list = read_noun_list()
possessive_noun_list = extract_possessive_examples(
    sentences, possessive_noun_list
)

In [10]:
# 프롬프트 5로 Codex(GPT-6.1 Sol High)가 생성한 코드
def extract_s_possessive_examples(corpus_data, noun_list):
    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사의 의존소 X 확인
            for x in dependents.get(token[3], []):
                if x[6] != "nmod:poss(소유 수식어)":
                    continue

                # X의 의존소 하나가 두 조건을 모두 만족하는지 확인
                has_s_marker = any(
                    dependent[2].lower() == "'s"
                    and dependent[6] == "case(격)"
                    for dependent in dependents.get(x[3], [])
                )

                if has_s_marker:
                    noun_list[lemma].append(
                        (sentence_id, x[0], sentence_text)
                    )

    return noun_list


# 새 딕셔너리로 시작하여 이전 추출 결과와 섞이지 않도록 함
s_possessive_noun_list = read_noun_list()
s_possessive_noun_list = extract_s_possessive_examples(
    sentences, s_possessive_noun_list
)

In [11]:
# 프롬프트 6으로 Codex(GPT-6.1 Sol High)가 생성한 코드
def extract_of_examples(corpus_data, noun_list):
    excluded_lemmas = {"kind", "sort", "type"}

    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사의 의존소 X 확인
            for x in dependents.get(token[3], []):
                if x[6] != "nmod(명사형 수식어)":
                    continue

                if x[1] in {"SYM", "SYM(기호)"}:
                    continue

                if x[2].lower() in excluded_lemmas:
                    continue

                # X의 의존소 하나가 of와 case 조건을 모두 만족하는지 확인
                has_of_marker = any(
                    dependent[2].lower() == "of"
                    and dependent[6] == "case(격)"
                    for dependent in dependents.get(x[3], [])
                )

                if has_of_marker:
                    noun_list[lemma].append(
                        (sentence_id, x[0], sentence_text)
                    )

    return noun_list


# 새 딕셔너리로 시작하여 이전 추출 결과와 섞이지 않도록 함
of_noun_list = read_noun_list()
of_noun_list = extract_of_examples(sentences, of_noun_list)

In [12]:
# 프롬프트 7로 Codex(GPT-6.1 Sol High)가 생성한 코드
def extract_whose_examples(corpus_data, noun_list):
    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 같은 의존소에 대해 두 조건을 모두 확인
            for dependent in dependents.get(token[3], []):
                if (
                    dependent[6] == "nmod:poss(소유 수식어)"
                    and dependent[0].lower() == "whose"
                ):
                    noun_list[lemma].append(
                        (sentence_id, dependent[0], sentence_text)
                    )

    return noun_list


# 새 딕셔너리로 시작하여 이전 추출 결과와 섞이지 않도록 함
whose_noun_list = read_noun_list()
whose_noun_list = extract_whose_examples(sentences, whose_noun_list)

In [13]:
# 프롬프트 4,5,6,7을 수정해 Codex(GPT-6.1 Sol High)가 다시 생성한 코드
def extract_possessive_examples(corpus_data, noun_list):
    possessives = {"my", "your", "his", "her", "its", "our", "their"}

    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사를 핵어로 갖는 의존소 확인
            for dependent in dependents.get(token[3], []):
                if (
                    dependent[6] == "nmod:poss"
                    and dependent[0].lower() in possessives
                ):
                    noun_list[lemma].append(
                        (sentence_id, dependent[0], sentence_text)
                    )

    return noun_list


def extract_s_possessive_examples(corpus_data, noun_list):
    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사의 의존소 X 확인
            for x in dependents.get(token[3], []):
                if x[6] != "nmod:poss":
                    continue

                # X의 의존소 하나가 두 조건을 모두 만족하는지 확인
                has_s_marker = any(
                    dependent[2].lower() == "'s"
                    and dependent[6] == "case"
                    for dependent in dependents.get(x[3], [])
                )

                if has_s_marker:
                    noun_list[lemma].append(
                        (sentence_id, x[0], sentence_text)
                    )

    return noun_list


def extract_of_examples(corpus_data, noun_list):
    excluded_lemmas = {"kind", "sort", "type"}

    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 목록 명사의 의존소 X 확인
            for x in dependents.get(token[3], []):
                if x[6] != "nmod":
                    continue

                if x[1] == "SYM":
                    continue

                if x[2].lower() in excluded_lemmas:
                    continue

                # X의 의존소 하나가 of와 case 조건을 모두 만족하는지 확인
                has_of_marker = any(
                    dependent[2].lower() == "of"
                    and dependent[6] == "case"
                    for dependent in dependents.get(x[3], [])
                )

                if has_of_marker:
                    noun_list[lemma].append(
                        (sentence_id, x[0], sentence_text)
                    )

    return noun_list


def extract_whose_examples(corpus_data, noun_list):
    for sentence_id, tokens in corpus_data:
        sentence_text = " ".join(token[0] for token in tokens)

        # 핵어 ID별로 의존소를 모음
        dependents = {}
        for token in tokens:
            dependents.setdefault(token[4], []).append(token)

        for token in tokens:
            lemma = token[2].lower()

            if token[1] != "NOUN" or lemma not in noun_list:
                continue

            # 같은 의존소에 대해 두 조건을 모두 확인
            for dependent in dependents.get(token[3], []):
                if (
                    dependent[6] == "nmod:poss"
                    and dependent[0].lower() == "whose"
                ):
                    noun_list[lemma].append(
                        (sentence_id, dependent[0], sentence_text)
                    )

    return noun_list


# 각각 새 딕셔너리로 시작하여 이전 추출 결과와 섞이지 않도록 함
possessive_noun_list = read_noun_list()
possessive_noun_list = extract_possessive_examples(
    sentences, possessive_noun_list
)

s_possessive_noun_list = read_noun_list()
s_possessive_noun_list = extract_s_possessive_examples(
    sentences, s_possessive_noun_list
)

of_noun_list = read_noun_list()
of_noun_list = extract_of_examples(sentences, of_noun_list)

whose_noun_list = read_noun_list()
whose_noun_list = extract_whose_examples(sentences, whose_noun_list)

In [15]:
# 프롬프트 8로 Codex(GPT-6.1 Sol High)가 생성한 코드
from pathlib import Path


def save_noun_examples(noun_examples_dict, folder, header, type):
    output_folder = Path(folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    count_path = output_folder / f"{header}_noun_{type}_count.txt"
    examples_path = output_folder / f"{header}_noun_{type}.txt"

    with (
        count_path.open("w", encoding="utf-8") as count_file,
        examples_path.open("w", encoding="utf-8") as examples_file
    ):
        for noun, examples in noun_examples_dict.items():
            if not examples:
                continue

            # 튜플 전체를 기준으로 중복 제거, 처음 등장한 순서 유지
            unique_examples = list(dict.fromkeys(examples))

            # 빈도수 파일: 명사, 중복 제거한 용례 수, 첫 번째 예문
            first_sentence = unique_examples[0][2]
            count_file.write(
                f"{noun}\t{len(unique_examples)}\t{first_sentence}\n"
            )

            # 용례 파일: 명사, 문장 ID, 소유자 단어, 전체 문장
            for sentence_id, owner, sentence in examples:
                examples_file.write(
                    f"{noun}\t{sentence_id}\t{owner}\t{sentence}\n"
                )

    return str(count_path), str(examples_path)


# 사용 예: 소유 대명사 용례 저장
count_file, examples_file = save_noun_examples(
    noun_examples_dict=possessive_noun_list,
    folder="/content/results",
    header="bnc",
    type="pron"
)

In [16]:
# 프롬프트 9로 Codex(GPT-6.1 Sol High)가 생성한 코드
import json
from pathlib import Path


def main(
    corpus_filename="/content/bnc_tagged.txt",
    noun_filename="/content/noun_list.txt"
):
    output_folder = Path("output")
    output_folder.mkdir(parents=True, exist_ok=True)

    # 1. 코퍼스와 명사 목록 전처리
    corpus_data = read_bnc_tagged(corpus_filename)
    noun_dictionary = read_noun_list(noun_filename)

    # 2. 문장 ID와 문장 텍스트를 리스트로 저장
    corpus_text_data = [
        (sentence_id, " ".join(token[0] for token in tokens))
        for sentence_id, tokens in corpus_data
    ]

    with (output_folder / "bnc_corpus_data.txt").open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(corpus_text_data, f, ensure_ascii=False, indent=2)

    # 3. 용례를 추가하기 전의 명사 딕셔너리 저장
    with (output_folder / "noun_dictionary.txt").open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(noun_dictionary, f, ensure_ascii=False, indent=2)

    # 문장 텍스트 저장용 임시 리스트의 메모리 해제
    del corpus_text_data

    # 4. 유형별 추출 함수 지정
    extractors = [
        ("all", extract_noun_examples),
        ("pron", extract_possessive_examples),
        ("s", extract_s_possessive_examples),
        ("of", extract_of_examples),
        ("whose", extract_whose_examples),
    ]

    # 5. 유형마다 새 명사 딕셔너리로 추출하고 저장
    for example_type, extractor in extractors:
        fresh_noun_dictionary = read_noun_list(noun_filename)
        noun_examples = extractor(corpus_data, fresh_noun_dictionary)

        save_noun_examples(
            noun_examples_dict=noun_examples,
            folder=str(output_folder),
            header="bnc",
            type=example_type
        )

        print(f"{example_type} 용례 및 빈도수 파일 저장 완료")

    print(f"총 12개 파일 저장 완료: {output_folder.resolve()}")


# 실행
if __name__ == "__main__":
    main()

전체 문장 수: 94171
시작 태그가 없던 문장 수: 2598
all 용례 및 빈도수 파일 저장 완료
pron 용례 및 빈도수 파일 저장 완료
s 용례 및 빈도수 파일 저장 완료
of 용례 및 빈도수 파일 저장 완료
whose 용례 및 빈도수 파일 저장 완료
총 12개 파일 저장 완료: /content/output
